# Spark in a Microsoft Fabric notebook

This notebook runs in Microsoft Fabric, with the **Big_Data** lakehouse attached. It loads two sources that are already in that lakehouse and analyzes them with Spark.

| Source | Where it lives | How Spark reads it |
| --- | --- | --- |
| Public holidays table | `Tables / dbo / publicholidays` | `spark.read.synapsesql` through the SQL analytics endpoint |
| People file | `Files / sample_datasets / sample_dataset.parquet` | `spark.read.parquet` |

The holiday table follows the [Azure Open Datasets public holidays](https://learn.microsoft.com/en-us/azure/open-datasets/dataset-public-holidays) layout: `countryOrRegion`, `countryRegionCode`, `date`, `holidayName`, `isPaidTimeOff`, `normalizeHolidayName`.

The Parquet file is the 100,000-row people file from `data/sample_dataset.parquet` (registration dates from 2016-10-05 through 2026-10-05).

Fabric starts `spark` when the session comes up. Cells below use that session. Attach **Big_Data** as the default lakehouse before you run the notebook.

## 1. Load the lakehouse table

Run the connector imports before `synapsesql`. The argument is a three-part name: lakehouse, schema, table. This reads `publicholidays` from the **Big_Data** SQL analytics endpoint in the same workspace.

In [ ]:
import com.microsoft.spark.fabric
from com.microsoft.spark.fabric.Constants import Constants

df = spark.read.synapsesql("Big_Data.dbo.publicholidays")
display(df)

## Name the holiday table

The load cell stores a plan in `df`. This cell copies that plan to `holidays`, fixes the column names if the table returns them in a different case, and registers the SQL view `holidays`. The queries below use that name.

In [ ]:
expected = {
    "countryorregion": "countryOrRegion",
    "countryregioncode": "countryRegionCode",
    "date": "date",
    "holidayname": "holidayName",
    "ispaidtimeoff": "isPaidTimeOff",
    "normalizeholidayname": "normalizeHolidayName",
}
lower_to_actual = {column.lower(): column for column in df.columns}
missing = [name for key, name in expected.items() if key not in lower_to_actual]
if missing:
    print("Columns on Big_Data.dbo.publicholidays:", df.columns)
    raise ValueError(
        "Expected the Azure Open Datasets public-holidays columns. Missing: "
        + ", ".join(missing)
    )

holidays = df
for key, canonical in expected.items():
    actual = lower_to_actual[key]
    if actual != canonical:
        holidays = holidays.withColumnRenamed(actual, canonical)

holidays.createOrReplaceTempView("holidays")
holidays.printSchema()

## Describe the holiday table

`describe()` prints **count**, **mean**, **stddev**, **min**, and **max**. It covers string and numeric columns. On a string, mean and standard deviation are empty, and min and max are the first and last values in alphabetical order. Date and boolean columns are omitted, so this call keeps the name columns. It reads the `holidays` frame.

In [ ]:
display(
    holidays.select("countryOrRegion", "countryRegionCode", "holidayName").describe()
)

## 2. Simple Spark SQL

Each query reads the `holidays` view. These use projection, filter, grouping, and aggregates.

In [ ]:
display(spark.sql("""
    SELECT
        countryOrRegion,
        COUNT(*) AS holidays
    FROM holidays
    GROUP BY countryOrRegion
    ORDER BY holidays DESC, countryOrRegion
"""))

In [ ]:
display(spark.sql("""
    SELECT
        countryRegionCode,
        `date`,
        holidayName,
        isPaidTimeOff
    FROM holidays
    WHERE countryOrRegion = 'United States'
      AND YEAR(`date`) = 2026
    ORDER BY `date`
"""))

In [ ]:
display(spark.sql("""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT countryOrRegion) AS countries,
        COUNT(DISTINCT holidayName) AS holiday_names,
        MIN(`date`) AS earliest,
        MAX(`date`) AS latest
    FROM holidays
"""))

## 3. Choose which rows to keep

Section 2 filtered one country and one year. These queries add three more filters. `IN` keeps a list of countries. `LIKE` keeps holiday names that contain a word. `IS NULL` keeps rows where paid time off was not filled in.

In [ ]:
display(spark.sql("""
    SELECT
        countryOrRegion,
        `date`,
        holidayName,
        isPaidTimeOff
    FROM holidays
    WHERE YEAR(`date`) = 2026
      AND countryOrRegion IN ('United States', 'Canada', 'United Kingdom')
    ORDER BY countryOrRegion, `date`
"""))

In [ ]:
display(spark.sql("""
    SELECT
        countryOrRegion,
        `date`,
        holidayName
    FROM holidays
    WHERE YEAR(`date`) = 2026
      AND holidayName LIKE '%Christmas%'
    ORDER BY `date`, countryOrRegion
"""))

In [ ]:
display(spark.sql("""
    SELECT
        countryOrRegion,
        COUNT(*) AS holidays
    FROM holidays
    WHERE isPaidTimeOff IS NULL
    GROUP BY countryOrRegion
    ORDER BY holidays DESC, countryOrRegion
"""))

## 4. Keep some groups, and stop after a few rows

`WHERE` drops rows before the count. `HAVING` drops groups after the count. `LIMIT` stops the result after the first rows of the sort.

In [ ]:
display(spark.sql("""
    SELECT
        countryOrRegion,
        COUNT(*) AS holidays
    FROM holidays
    WHERE YEAR(`date`) = 2026
    GROUP BY countryOrRegion
    HAVING COUNT(*) >= 12
    ORDER BY holidays DESC, countryOrRegion
"""))

In [ ]:
display(spark.sql("""
    SELECT
        countryOrRegion,
        COUNT(*) AS holidays
    FROM holidays
    WHERE YEAR(`date`) = 2026
    GROUP BY countryOrRegion
    ORDER BY holidays DESC, countryOrRegion
    LIMIT 8
"""))

## 5. Read the date

`BETWEEN` keeps a range of dates. `MONTH` and `YEAR` pull a number out of the date so you can group by it.

In [ ]:
display(spark.sql("""
    SELECT
        `date`,
        holidayName,
        isPaidTimeOff
    FROM holidays
    WHERE countryOrRegion = 'United States'
      AND `date` BETWEEN '2026-01-01' AND '2026-06-30'
    ORDER BY `date`
"""))

In [ ]:
display(spark.sql("""
    SELECT
        MONTH(`date`) AS month,
        COUNT(*) AS holidays
    FROM holidays
    WHERE countryOrRegion = 'United States'
      AND YEAR(`date`) = 2026
    GROUP BY MONTH(`date`)
    ORDER BY month
"""))

In [ ]:
display(spark.sql("""
    SELECT
        YEAR(`date`) AS year,
        COUNT(*) AS holidays,
        COUNT(DISTINCT countryOrRegion) AS countries
    FROM holidays
    GROUP BY YEAR(`date`)
    ORDER BY year
"""))

## 6. Load the Parquet file

`Files/...` is relative to the attached lakehouse. The file uploaded from `data/sample_dataset.parquet` is `Files/sample_datasets/sample_dataset.parquet`.

| Column | Type | Meaning |
| --- | --- | --- |
| `full_name` | string | Synthetic person name |
| `email` | string | Synthetic email |
| `city` | string | Synthetic city |
| `job` | string | Job title |
| `registered_on` | date | Registration date |
| `salary` | int | Annual salary (28,000–220,000) |
| `years_of_experience` | int | Years of experience (0–40) |
| `gender` | string | Female, Male, or Non-binary |
| `latitude`, `longitude` | double | Synthetic coordinates |

In [ ]:
PARQUET_PATH = "Files/sample_datasets/sample_dataset.parquet"

people_raw = spark.read.parquet(PARQUET_PATH)
print(f"{people_raw.count():,} rows from {PARQUET_PATH}")
people_raw.printSchema()
display(people_raw.limit(10))

## Describe the people file

`describe()` is the same five statistics as on the holiday names. `gender` is a string, so the useful rows there are count, min, and max, and those min and max values are alphabetical.

`summary()` adds the quartiles. The **50%** row is the median salary and the median years of experience. The median sits at the middle of the values, which is the figure to read when a few high salaries pull the mean up.

In [ ]:
display(people_raw.select("salary", "years_of_experience", "gender").describe())
display(people_raw.select("salary", "years_of_experience").summary())

## 7. Reshape the people data

The summaries above describe the raw file. This section adds the registration year, an experience band, and a salary band, then lists senior high earners and pay by experience band.

`people.cache()` stores this frame after the first `count()`. The SQL in the next sections reads it, so later actions reuse that copy.

In [ ]:
from pyspark.sql import functions as F

people = (
    people_raw
    .withColumn("registered_year", F.year("registered_on"))
    .withColumn("registered_month", F.month("registered_on"))
    .withColumn(
        "experience_band",
        F.when(F.col("years_of_experience") < 5, "0-4 years")
        .when(F.col("years_of_experience") < 10, "5-9 years")
        .when(F.col("years_of_experience") < 20, "10-19 years")
        .otherwise("20+ years"),
    )
    .withColumn(
        "band_order",
        F.when(F.col("years_of_experience") < 5, 1)
        .when(F.col("years_of_experience") < 10, 2)
        .when(F.col("years_of_experience") < 20, 3)
        .otherwise(4),
    )
    .withColumn(
        "salary_band",
        F.when(F.col("salary") < 60000, "Under 60k")
        .when(F.col("salary") < 100000, "60k-99k")
        .when(F.col("salary") < 150000, "100k-149k")
        .otherwise("150k+"),
    )
)


people.createOrReplaceTempView("people")
print(f"{people.count():,} people cached as view 'people'")

seniors = (
    people
    .filter((F.col("years_of_experience") >= 20) & (F.col("salary") >= 150000))
    .select(
        "full_name",
        "job",
        "city",
        "years_of_experience",
        "salary",
        "registered_on",
    )
    .orderBy(F.desc("salary"))
)
print(f"Senior high earners (20+ years and salary >= 150k): {seniors.count():,}")
display(seniors.limit(15))

In [ ]:
by_experience = (
    people.groupBy("band_order", "experience_band")
    .agg(
        F.count("*").alias("people"),
        F.round(F.avg("salary"), 0).alias("avg_salary"),
        F.expr("percentile_approx(salary, 0.5)").alias("median_salary"),
        F.min("salary").alias("min_salary"),
        F.max("salary").alias("max_salary"),
    )
    .orderBy("band_order")
    .drop("band_order")
)
display(by_experience)

by_gender = (
    people.groupBy("gender")
    .agg(
        F.count("*").alias("people"),
        F.round(F.avg("salary"), 0).alias("avg_salary"),
        F.expr("percentile_approx(salary, 0.5)").alias("median_salary"),
    )
    .orderBy(F.desc("people"))
)
display(by_gender)

## 8. Simple Spark SQL on the people file

Same questions as the DataFrame aggregations above, written in SQL against the `people` view. After the year query, switch the output from Table to Chart if you want a bar of registrations by year.

In [ ]:
display(spark.sql("""
    SELECT
        gender,
        COUNT(*) AS people,
        ROUND(AVG(salary), 0) AS avg_salary,
        percentile_approx(salary, 0.5) AS median_salary
    FROM people
    GROUP BY gender
    ORDER BY people DESC
"""))

In [ ]:
display(spark.sql("""
    SELECT
        registered_year,
        COUNT(*) AS registrations,
        ROUND(AVG(salary), 0) AS avg_salary
    FROM people
    GROUP BY registered_year
    ORDER BY registered_year
"""))

## 9. More queries on the people file

These stay on one table. `LIMIT` keeps the first rows after the sort. The city query uses `WHERE` before `GROUP BY`, so it counts only people whose salary is at least 100,000.

In [ ]:
display(spark.sql("""
    SELECT
        job,
        COUNT(*) AS people,
        ROUND(AVG(salary), 0) AS avg_salary,
        MIN(salary) AS min_salary,
        MAX(salary) AS max_salary
    FROM people
    GROUP BY job
    ORDER BY avg_salary DESC, job
    LIMIT 10
"""))

In [ ]:
display(spark.sql("""
    SELECT
        experience_band,
        COUNT(*) AS people,
        ROUND(AVG(salary), 0) AS avg_salary
    FROM people
    GROUP BY experience_band
    ORDER BY MIN(years_of_experience)
"""))

In [ ]:
display(spark.sql("""
    SELECT
        city,
        COUNT(*) AS people,
        ROUND(AVG(salary), 0) AS avg_salary
    FROM people
    WHERE salary >= 100000
    GROUP BY city
    ORDER BY people DESC, city
    LIMIT 10
"""))